In [1]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt

import os
from glob import glob

## Colocation dataset
- pd dataframe
- each row corresponds to a colocation, and contained terms involved in the conservation equation and additional information that will be used as coordinates (e.g. longitude, latitude, distance to coast etc ...) 

In [39]:
from diagnosis import one_comb

base_dic0 = dict(
    dt="12h",
    drifter_preprocess="",
    drifter_preprocess_param="",
    alti_product_key="swot250m",
    alti_diff_method="fromduacsv",
    alti_diff_method_param="",
    ggd_var="fromduacsv",
    wd_product_key="era5",
    wd_model="rio",
    wd_depth="0",
)


df, id_comb = one_comb(**base_dic0)

df

12h_spectral_decomp_all_med_variational_10min_v1____swot250m_fromduacsv__fromduacsv__era5_rio0


,datetime,longitude,latitude,pass_number,depth,time_to_swot,cycle_number,cycle_date,drifter_id,drifter_type,accn,acce,core,corn,ggde,ggdn,phi,distance_to_coast,wde,wdn
row_number,,,,,,,,,,,,,,,,,,,,
0,2023-03-28 12:30:00,5.672100,42.931158,3,0,0 days 11:50:02.155782592,474,2023-03-29 00:20:02.155782592,0-4388554,CARTHE,1.556294e-06,-1.022971e-06,-0.000007,-0.000021,0.000005,0.000040,0.253373,17949.549787,0.000002,-7.748975e-07
1,2023-03-28 12:40:00,5.670559,42.931511,3,0,0 days 11:40:02.155782592,474,2023-03-29 00:20:02.155782592,0-4388554,CARTHE,1.623461e-06,-9.972787e-07,-0.000007,-0.000021,0.000015,0.000041,0.252726,17953.760494,0.000001,-8.200891e-07
2,2023-03-28 12:50:00,5.669013,42.931870,3,0,0 days 11:30:02.155782592,474,2023-03-29 00:20:02.155782592,0-4388554,CARTHE,1.699765e-06,-9.185723e-07,-0.000007,-0.000021,0.000016,0.000041,0.252542,17943.992390,0.000001,-8.651695e-07
3,2023-03-28 13:00:00,5.667463,42.932234,3,0,0 days 11:20:02.155782592,474,2023-03-29 00:20:02.155782592,0-4388554,CARTHE,1.773407e-06,-7.681015e-07,-0.000007,-0.000021,0.000016,0.000040,0.252899,17966.495686,0.000001,-9.101388e-07
4,2023-03-28 13:10:00,5.665910,42.932603,3,0,0 days 11:10:02.155782592,474,2023-03-29 00:20:02.155782592,0-4388554,CARTHE,1.824080e-06,-5.274647e-07,-0.000007,-0.000021,0.000012,0.000040,0.253310,17935.147936,0.000001,-9.223292e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377476,2023-07-10 02:30:00,1.703455,38.389120,16,15,0 days 07:33:38.131544384,577,2023-07-09 18:56:21.868455616,300534060218400,SVP,-1.683142e-06,-1.636717e-05,0.000029,-0.000016,-0.000013,-0.000014,2.912227,31000.000000,-0.000003,4.685282e-06
377477,2023-07-10 02:40:00,1.702295,38.387378,16,15,0 days 07:43:38.131544384,577,2023-07-09 18:56:21.868455616,300534060218400,SVP,-7.520432e-08,-1.606230e-05,0.000029,-0.000017,-0.000013,-0.000004,2.911467,31107.922744,-0.000003,4.924376e-06
377478,2023-07-10 02:50:00,1.701070,38.385633,16,15,0 days 07:53:38.131544384,577,2023-07-09 18:56:21.868455616,300534060218400,SVP,1.535037e-06,-1.587103e-05,0.000029,-0.000018,-0.000021,0.000017,2.910488,31803.002698,-0.000003,5.163632e-06


## Apply the statistical balance diagnosis method
### Compute sum of all conservation equation terms, 2 by 2 terms products, and sums excepting on term

In [69]:
def create_prod_exc_sum(
    df,
    id_comb,
    eqterms=["acc", "cor", "ggd", "wd"],
    dirsuf=["e", "n"],
    coords_list=[
        "datetime",
        "longitude",
        "latitude",
        "pass_number",
        "time_to_swot",
        "cycle_number",
        "drifter_id",
        "drifter_type",
        "depth",
        "phi",
        "distance_to_coast",
    ],
    null_mean=False,
    row_index="row_number",
):
    """
    Add sum of all conservation equation terms, 2 by 2 terms products, and sums excepting on term

    Parameters
    ----------
    df: `pandas.DataFrame` or `xr.Dataset`
        Input dataframe, each row corresponds to a colocation, and contains terms involved in the conservation equation and additional information that will be used as coordinates (e.g. longitude, latitude, distance to coast etc ...)
    id_comb: str,
            name of the reconstruction (for eventual concatenation with others)
    eqterms: list,
            keys for the terms of the equation
    dirsuf: list,
            suffixes indicating recontruction direction(s) (e.g. ['e', 'n'] for eastward and northward 2D reconstructions)
    coordlist : list,
            keys for additional information that will be used as coordinates (e.g. longitude, latitude, distance to coast etc ...)
    nullmean : bool,
            remove mean to equation terms prior to computation, default is True, but False might be useful for preliminary studies
    row_index : key for the colocation direction (row) if xarray dataset
    Returns
    -------
    ds: `xarray.Dataset`
        Output dataset with sum of all conservation equation terms, 2 by 2 terms products, and sums excepting on term added.

    """

    for direction in dirsuf:
        l = {eqterm: eqterm + direction for eqterm in eqterms}

        if isinstance(df, pd.DataFrame):
            ds = df[coords_list].to_xarray()
            ds["id_comb"] = id_comb
            ds = ds.set_coords("id_comb").expand_dims(dim="id_comb")
            if null_mean:
                for v in l:
                    df[l[v]] = df[l[v]] - df[l[v]].mean()
        if isinstance(df, xr.Dataset):
            ds = df[coords_list]
            if null_mean:
                for v in l:
                    df[l[v]] = df[l[v]] - df[l[v]].mean(row_index)

        # sum
        ds["sum" + direction] = sum([df[l[v]] for v in l])

        # simple var + sum- one term
        for v in l:
            ds[l[v]] = df[l[v]]
            ds["exc" + direction + "_" + v] = ds["sum" + direction] - ds[l[v]]

        # 2 by 2 product
        import itertools

        couple = list(itertools.combinations(list(l.keys()), 2))
        for c in couple:
            ds["prod" + direction + "_" + "_".join(c)] = (
                ds[c[0] + direction] * ds[c[1] + direction]
            )

    return ds

### Compute diagnosis variables

In [70]:
eqterms = ["acc", "cor", "ggd", "wd"]


def define_closure_vars(eqterms):
    # Mean Square in capital letter, mean square of the sum 'S', and sum of mean square 'sigma'
    closure_vars = [eqt.upper() + "*" for eqt in eqterms] + ["S*", "sigma*"]
    # Balanced and residual contribution
    closure_vars += ["B*_" + eqt for eqt in eqterms]
    closure_vars += ["E*_" + eqt for eqt in eqterms]
    # paired contribution
    # 2 by 2 product
    import itertools

    couple = list(itertools.combinations(eqterms, 2))
    closure_vars += ["X*_" + c[0] + "_" + c[1] for c in couple]
    return closure_vars


define_closure_vars(eqterms)

['ACC*',
 'COR*',
 'GGD*',
 'WD*',
 'S*',
 'sigma*',
 'B*_acc',
 'B*_cor',
 'B*_ggd',
 'B*_wd',
 'E*_acc',
 'E*_cor',
 'E*_ggd',
 'E*_wd',
 'X*_acc_cor',
 'X*_acc_ggd',
 'X*_acc_wd',
 'X*_cor_ggd',
 'X*_cor_wd',
 'X*_ggd_wd']

In [77]:
def effective_degree_number_of_nearest_coloc(ds):
    return len(
        ds.isel(id_comb=0)
        .to_dataframe()
        .groupby(["drifter_id", "cycle_number", "pass_number"], observed=False)
        .count()
    )


def compute_mean_square(
    ds,
    eqterms=["acc", "cor", "ggd", "wd"],
    dirsuf=["e", "n"],
    row_index="row_number",
    compute_error="no",
    effective_degree_function=effective_degree_number_of_nearest_coloc,
    vars_errors=None,
    attrs=True,
):
    """Compute closure stats
    ds : dataset containing terms values for all row_numbers
    dirsuf : directions of the reconstruction
    """
    closure_vars_ = define_closure_vars(eqterms)

    var = eqterms + ["sum"]
    VAR = [eqt.upper() for eqt in eqterms] + ["S"]

    # MEAN SQUARE
    dss = (ds[sum([[v + direction for v in var] for direction in dirsuf], [])]) ** 2
    for direction in dirsuf:
        dss = dss.rename(
            {var[i] + direction: VAR[i] + direction for i in range(len(var))}
        )
        dss["sigma" + direction] = sum(
            [dss[v + direction] for v in [eqt.upper() for eqt in eqterms]]
        )

    nb_coloc = len(ds[row_index])

    # Balanced and error
    for direction in dirsuf:
        for v in eqterms:
            dss["B" + direction + "_" + v] = -(
                (ds[v + direction] * ds["exc" + direction + "_" + v])
            )
            dss["E" + direction + "_" + v] = ds[v + direction] * ds["sum" + direction]

    # pairs contributions
    for v in [v for v in ds if "prod" in v]:
        dss[v.replace("prod", "X")] = -2 * ds[v]

    # Cyclo/anticyclo contribution
    # for direction in [d0, d1]:
    #    dss['D'+direction+'_cyclo'] = -2 *(ds['acc'+direction]+ds['cor'+direction])*ds['ggd'+direction]
    #    dss['D'+direction+'_anticyclo'] = -2 *(ds['acc'+direction]+ds['ggd'+direction])*ds['cor'+direction]

    # Sum of both direction
    if len(dirsuf) == 2:
        for v in closure_vars_:
            dss[v.replace("*", "")] = (
                dss[v.replace("*", dirsuf[0])] + dss[v.replace("*", dirsuf[1])]
            )

    # Mean
    dsm = dss.mean(row_index)

    # STATISTICAL ERRORS
    # central limit
    if compute_error == "centrallimit":
        # effective freedom degree (MIGHT BE MODIFIED FOR OTHER ESTIMATIONS)
        effective_degree = effective_degree_function(ds)
        print(effective_degree)
        # dsme = (2*dss.std('row_number')/np.sqrt(nb_coloc)).rename({v:'er__'+v for v in list(dss.keys())})
        dsme = (2 * dss.std(row_index) / np.sqrt(effective_degree)).rename(
            {v: "er__" + v for v in list(dss.keys())}
        )
        dsm = xr.merge([dsm, dsme])

    # bootstrap
    if compute_error == "bootstrap":
        if vars_errors == None:  # then compute for all
            vars_errors = sum(
                [
                    [v.replace("*", direction) for v in closure_vars]
                    for direction in dirsuf
                ],
                [],
            )

        D = []
        for id_ in dss.id_comb:
            dsme = xr.Dataset()
            dsme["id_comb"] = id_
            dsme = dsme.set_coords("id_comb").expand_dims("id_comb")
            for v in vars_errors:
                dsme[v] = (
                    "id_comb",
                    [compute_bootstrap_error(dss.sel(id_comb=id_)[v])],
                )
                print(v)
            D.append(dsme)
            print(id_)
        dsme = xr.concat(D, dim="id_comb").rename({v: "er__" + v for v in vars_errors})
        dsm = xr.merge([dsm, dsme])

    dsm.attrs["error_method"] = compute_error

    if attrs:
        dsm = assign_attrs(dsm, dirsuf + [""])
    return dsm


def compute_balance_stats(
    ds,
    eqterms=["acc", "cor", "ggd", "wd"],
    dirsuf=["e", "n"],
    row_index="row_number",
    compute_error="no",
    effective_degree_function=effective_degree_number_of_nearest_coloc,
    vars_errors=None,
    attrs=True,
):
    """Compute closure stats
    ds : dataset containing terms values for all row_numbers
    dirsuf : directions of the reconstruction
    """
    # Ensure mean is set to zero (compute variance)
    ds = create_prod_exc_sum(
        ds,
        id_comb="",
        eqterms=eqterms,
        dirsuf=dirsuf,
        coords_list=[],
        null_mean=True,
        row_index="row_number",
    )
    return ds
    # MS = var as mean are null
    # dss = compute_mean_square(ds, dirsuf, compute_error, vars_errors)
    # return dss


def assign_attrs(ds, dirsuf=["e", "n", ""]):
    """Assign attributes to compute_mean_square() functions dataset output"""
    for dir_ in dirsuf:
        if dir_ == "e":
            Dir = "Zonal"
        if dir_ == "n":
            Dir = "Meridional"
        if dir_ == "x":
            Dir = "Cross-track"
        if dir_ == "y":
            Dir = "Along-track"
        if dir_ == "":
            Dir = "Total"

        ds["ACC" + dir_] = ds["ACC" + dir_].assign_attrs(
            {"long_name": Dir + " Lagrangian acceleration MS"}
        )
        ds["COR" + dir_] = ds["COR" + dir_].assign_attrs(
            {"long_name": Dir + " Coriolis acceleration MS"}
        )
        ds["GGD" + dir_] = ds["GGD" + dir_].assign_attrs(
            {"long_name": Dir + " Pressure gradient term MS"}
        )
        ds["WD" + dir_] = ds["WD" + dir_].assign_attrs(
            {"long_name": Dir + " Wind term MS"}
        )

        ds["sigma" + dir_] = ds["sigma" + dir_].assign_attrs(
            {"long_name": Dir + r" $\Sigma$"}
        )
        ds["S" + dir_] = ds["S" + dir_].assign_attrs({"long_name": Dir + r" Residual"})

        ds["B" + dir_ + "_acc"] = ds["B" + dir_ + "_acc"].assign_attrs(
            {
                "long_name": Dir
                + r" Lagrangian acceleration balanced signal contribution"
            }
        )
        ds["B" + dir_ + "_cor"] = ds["B" + dir_ + "_cor"].assign_attrs(
            {"long_name": Dir + r" Coriolis acceleration balanced signal contribution"}
        )
        ds["B" + dir_ + "_ggd"] = ds["B" + dir_ + "_ggd"].assign_attrs(
            {"long_name": Dir + r" Pressure gradient term balanced signal contribution"}
        )
        ds["B" + dir_ + "_wd"] = ds["B" + dir_ + "_wd"].assign_attrs(
            {"long_name": Dir + r" Wind term balanced signal contribution"}
        )

        ds["E" + dir_ + "_acc"] = ds["E" + dir_ + "_acc"].assign_attrs(
            {"long_name": Dir + r" Lagrangian acceleration residual contribution"}
        )
        ds["E" + dir_ + "_cor"] = ds["E" + dir_ + "_cor"].assign_attrs(
            {"long_name": Dir + r" Coriolis acceleration residual contribution"}
        )
        ds["E" + dir_ + "_ggd"] = ds["E" + dir_ + "_ggd"].assign_attrs(
            {"long_name": Dir + r" Pressure gradient term residual contribution"}
        )
        ds["E" + dir_ + "_wd"] = ds["E" + dir_ + "_wd"].assign_attrs(
            {"long_name": Dir + r" Wind term residual contribution"}
        )

        ds["X" + dir_ + "_acc_cor"] = ds["X" + dir_ + "_acc_cor"].assign_attrs(
            {"long_name": Dir + r" Inertial balance contribution"}
        )
        ds["X" + dir_ + "_acc_ggd"] = ds["X" + dir_ + "_acc_ggd"].assign_attrs(
            {"long_name": Dir + r" Cyclostrophic contribution"}
        )
        ds["X" + dir_ + "_acc_wd"] = ds["X" + dir_ + "_acc_wd"].assign_attrs(
            {"long_name": Dir + r" Lagrangian - wind contribution"}
        )
        ds["X" + dir_ + "_cor_ggd"] = ds["X" + dir_ + "_cor_ggd"].assign_attrs(
            {"long_name": Dir + r" Geostrophic contribution"}
        )
        ds["X" + dir_ + "_cor_wd"] = ds["X" + dir_ + "_cor_wd"].assign_attrs(
            {"long_name": Dir + r" Coriolis - wind contribution"}
        )
        ds["X" + dir_ + "_ggd_wd"] = ds["X" + dir_ + "_ggd_wd"].assign_attrs(
            {"long_name": Dir + r" Pressure gradient - wind contribution"}
        )

    return ds

In [78]:
compute_mean_square(ds, compute_error="centrallimit")

3033


<xarray.Dataset> Size: 1kB
Dimensions:         (id_comb: 1)
Coordinates:
  * id_comb         (id_comb) <U94 376B '12h_spectral_decomp_all_med_variatio...
Data variables: (12/120)
    ACCe            (id_comb) float64 8B 1.572e-10
    CORe            (id_comb) float64 8B 2.926e-10
    GGDe            (id_comb) float64 8B 3.503e-10
    WDe             (id_comb) float64 8B 1.121e-10
    Se              (id_comb) float64 8B 3.895e-10
    ACCn            (id_comb) float64 8B 1.658e-10
    ...              ...
    er__X_acc_cor   (id_comb) float64 8B 3.339e-11
    er__X_acc_ggd   (id_comb) float64 8B 2.465e-11
    er__X_acc_wd    (id_comb) float64 8B 1.072e-11
    er__X_cor_ggd   (id_comb) float64 8B 3.977e-11
    er__X_cor_wd    (id_comb) float64 8B 2.191e-11
    er__X_ggd_wd    (id_comb) float64 8B 2.061e-11
Attributes:
    error_method:  centrallimit

In [79]:
compute_balance_stats(
    ds,
    eqterms=["acc", "cor", "ggd", "wd"],
    dirsuf=["e", "n"],
    compute_error="no",
    effective_degree_function=effective_degree_number_of_nearest_coloc,
    vars_errors=None,
    attrs=True,
)

<xarray.Dataset> Size: 47MB
Dimensions:        (row_number: 364164, id_comb: 1)
Coordinates:
  * row_number     (row_number) int64 3MB 0 1 2 3 ... 377478 377479 377480
  * id_comb        (id_comb) <U94 376B '12h_spectral_decomp_all_med_variation...
Data variables: (12/15)
    sumn           (id_comb, row_number) float64 3MB 2.18e-05 ... -1.37e-05
    accn           (id_comb, row_number) float64 3MB 1.618e-06 ... 4.864e-06
    excn_acc       (id_comb, row_number) float64 3MB 2.018e-05 ... -1.857e-05
    corn           (id_comb, row_number) float64 3MB -2.136e-05 ... -1.989e-05
    excn_cor       (id_comb, row_number) float64 3MB 4.315e-05 ... 6.188e-06
    ggdn           (id_comb, row_number) float64 3MB 4.281e-05 ... -3.597e-06
    ...             ...
    prodn_acc_cor  (id_comb, row_number) float64 3MB -3.455e-11 ... -9.675e-11
    prodn_acc_ggd  (id_comb, row_number) float64 3MB 6.927e-11 ... -1.75e-11
    prodn_acc_wd   (id_comb, row_number) float64 3MB -2.058e-12 ... 2.393e-11
    prodn_cor_ggd  (id_comb, row_number) float64 3MB -9.142e-10 ... 7.155e-11
    prodn_cor_wd   (id_comb, row_number) float64 3MB 2.716e-11 ... -9.787e-11
    prodn_ggd_wd   (id_comb, row_number) float64 3MB -5.445e-11 ... -1.77e-11